In [5]:
from bayesianquilts.data.rwa import get_data, item_keys, item_text, to_reverse, scale_indices
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

In [22]:
df, num_people = get_data(polars_out=True)

In [23]:
print(f"Dataset shape: {df.shape}")
print(f"Number of people: {num_people}")
print(f"Number of items: {len(item_keys)}")
print(f"Response categories: 0-8 (9 levels)")
print(f"\nScale 1 items ({len(scale_indices[0])} items): {[item_keys[i] for i in scale_indices[0]]}")
print(f"Scale 2 items ({len(scale_indices[1])} items): {[item_keys[i] for i in scale_indices[1]]}")

Dataset shape: (9881, 23)
Number of people: 9881
Number of items: 22
Response categories: 0-8 (9 levels)

Scale 1 items (12 items): ['Q2', 'Q4', 'Q6', 'Q8', 'Q9', 'Q11', 'Q13', 'Q15', 'Q16', 'Q18', 'Q20', 'Q21']
Scale 2 items (10 items): ['Q1', 'Q3', 'Q5', 'Q7', 'Q10', 'Q12', 'Q14', 'Q17', 'Q19', 'Q22']


In [24]:
SUBSAMPLE_N = 500
rng = np.random.default_rng(42)
idx = rng.choice(num_people, size=SUBSAMPLE_N, replace=False)
idx.sort()

sub_df = df[idx.tolist()]
print(f"Subsample size: {len(sub_df)}")

Subsample size: 500


In [26]:
sub_df.head()

person,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15,Q16,Q17,Q18,Q19,Q20,Q21,Q22
u32,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
51,0,0,0,8,1,7,0,7,7,4,8,2,8,6,5,0,2,8,1,8,8,1
64,1,0,2,8,0,8,1,5,8,1,8,1,8,6,6,0,0,8,0,7,4,3
69,0,0,2,8,0,8,0,8,8,2,8,0,8,0,8,0,6,8,0,7,4,0
96,7,5,8,1,8,6,5,2,2,7,1,6,1,8,1,6,8,5,6,0,0,7
136,4,2,1,7,5,7,2,7,1,1,7,3,7,1,5,0,1,6,1,7,4,2


In [8]:
df.head()

person,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15,Q16,Q17,Q18,Q19,Q20,Q21,Q22
u32,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,3,0,0,8,2,8,1,6,7,0,8,0,8,0,8,0,0,8,1,8,8,0
1,2,1,0,7,1,6,0,7,8,2,6,4,8,1,8,0,0,7,0,6,6,0
2,0,5,6,8,2,8,8,3,8,8,7,8,8,7,8,1,8,2,4,6,8,8
3,0,0,0,7,0,7,0,8,8,0,8,1,7,0,7,0,0,8,0,8,8,0
4,0,0,0,2,0,8,0,0,6,0,8,1,8,4,8,0,0,2,0,8,3,0


In [10]:
SUBSAMPLE_N = 500
rng = np.random.default_rng(42)
idx = rng.choice(num_people, size=SUBSAMPLE_N, replace=False)
idx.sort()
sub_df = df[idx.tolist()]
print(f"Subsample size: {len(sub_df)}")

Subsample size: 500


In [27]:
df_ys = pd.read_stata(
    "/home/luis/CGE/mice_imp/data/Datenexport_test_mar2026_2.dta",
    convert_categoricals=True,  # Stata labeled values -> pandas categorical
    convert_dates=True,         # Convert Stata dates
    preserve_dtypes=True        # Keep Stata-like dtypes when possible
)

In [28]:
df_ys.shape

(642, 213)

In [11]:
from bayesianquilts.imputation.mice_loo import MICEBayesianLOO

In [19]:
model = MICEBayesianLOO(
    n_imputations=5,
    max_iter=5,
    verbose=True,
    min_obs=20,
    pathfinder_maxiter=1000,
    pathfinder_num_samples=200,
    prior_scale=0.5, # Weekly informative prior for regularization
    batch_size=512,
    inference_method='advi'
)

In [35]:
df_ys = df_ys[["id_work"] + [c for c in df_ys.columns if c != "id_work"]]

# Convert categoricals to numeric codes and coerce object columns to float
for col in df_ys.columns:
    if hasattr(df_ys[col], 'cat'):
        df_ys[col] = df_ys[col].cat.codes.astype(float).where(df_ys[col].notna(), other=float('nan'))
    elif df_ys[col].dtype == object:
        df_ys[col] = pd.to_numeric(df_ys[col], errors='coerce')
df_ys = df_ys.astype(float)

In [36]:
df_ys[df_ys.columns[:3]]

,id_work,ALTER,Z_SRH
0,4.0,61.0,2.0
1,6.0,53.0,2.0
2,11.0,46.0,1.0
3,16.0,59.0,2.0
4,18.0,60.0,1.0
...,...,...,...
637,1347.0,60.0,2.0
638,1348.0,61.0,2.0
639,1351.0,63.0,2.0
640,1357.0,49.0,2.0


In [32]:
from pathlib import Path
output_path = Path("/home/luis/CGE/mice_imp/data")

In [37]:
model.fit_loo_models(df_ys, save_dir=output_path, n_jobs=10, n_top_features=200)

Saving incremental results to /home/luis/CGE/mice_imp/data
Computing feature correlations...
Fitting MICE Bayesian LOO-CV models with Pathfinder
  Variables: 213
  Observations: 642
  Min obs per model: 20
  Parallel jobs: 10
  Top features per target: 200
  Global Ordinal Values: [0. 1. 2. 3. 4. 5.] (n=6)

Fitting zero-predictor models...
  Scheduling 213 zero-predictor jobs...
    khat=inf > 0.7, discarding zero-predictor model
    khat=inf > 0.7, discarding zero-predictor model
  Var 0 (id_work): FAILED/SKIPPED
  Var 1 (ALTER): FAILED/SKIPPED
  Var 2 (Z_SRH): n_obs=637, elpd/n=-1.6740
  Var 3 (Z_SEX): n_obs=605, elpd/n=-0.6242
  Var 4 (Z_GRUND_RENTE): n_obs=610, elpd/n=-1.3447
  Var 5 (Z_GEHHILFE): n_obs=170, elpd/n=-0.6096
  Var 6 (PD438PD439): n_obs=635, elpd/n=-1.6116
  Var 7 (PD440_PD441): n_obs=636, elpd/n=-1.4440
  Var 8 (PDR012): n_obs=636, elpd/n=-1.3354
  Var 9 (PDR013): n_obs=635, elpd/n=-1.6531
  Var 10 (PDR016): n_obs=634, elpd/n=-1.4068
  Var 11 (PD354): n_obs=634, elpd

/home/luis/.cache/pypoetry/virtualenvs/mice-imp-toZTZ-oV-py3.12/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Var 72 (PD017): n_obs=59, elpd/n=-1.5657
  Var 73 (PD045): n_obs=59, elpd/n=-1.3736
  Var 74 (PD076): n_obs=64, elpd/n=-1.1256
  Var 75 (PD082): n_obs=59, elpd/n=-1.2643
  Var 76 (PD106_1): n_obs=59, elpd/n=-1.3256
  Var 77 (PD167): n_obs=64, elpd/n=-1.4690
  Var 78 (PD178): n_obs=65, elpd/n=-1.6281
  Var 79 (PD184): n_obs=59, elpd/n=-1.6214
  Var 80 (PD201): n_obs=59, elpd/n=-1.3112
  Var 81 (PD209): FAILED/SKIPPED
  Var 82 (PD436): n_obs=65, elpd/n=-1.3039
  Var 83 (PD437): n_obs=48, elpd/n=-1.7049
  Var 84 (PD447): n_obs=64, elpd/n=-1.7354
  Var 85 (PD448): n_obs=48, elpd/n=-1.3777
  Var 86 (PD450): n_obs=65, elpd/n=-1.6564
  Var 87 (PD452): n_obs=47, elpd/n=-1.6687
  Var 88 (PD454): n_obs=48, elpd/n=-2.0639
  Var 89 (PD457): n_obs=47, elpd/n=-1.8246
  Var 90 (PD245_142): n_obs=47, elpd/n=-1.5193
  Var 91 (PD461B): FAILED/SKIPPED
  Var 92 (PD462): n_obs=58, elpd/n=-1.5978
  Var 93 (PD465): n_obs=47, elpd/n=-1.2667
  Var 94 (PD466B): FAILED/SKIPPED
  Var 95 (PD499): n_obs=65, elpd/

In [38]:
model.save(output_path /"mice_bayesian_loo_model.yaml")